## Imports and Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, set_seed
from datasets import Dataset

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Dataset Loading (Training + Golden Standart)

In [ ]:
# --- 1. Training & Validation Set Preparation ---
print("Loading main dataset...")
df_main = pd.read_csv('kyrgyz_ready_for_ml.csv').dropna(subset=['clean_text', 'sentiment'])
df_main = df_main.rename(columns={'clean_text': 'text', 'sentiment': 'label'})

train_df, test_df = train_test_split(df_main, test_size=0.2, random_state=42, stratify=df_main['label'])

# --- 2. Golden Standard Preparation ---
print("Loading Golden Standard...")
df_golden = pd.read_excel('golden_test_set_300.xlsx')
df_golden = df_golden[~df_golden['true_human_label'].astype(str).str.lower().isin(['x', 'х', 'drop'])]
df_golden = df_golden.rename(columns={'true_human_label': 'label'})
df_golden['label'] = df_golden['label'].astype(int)
df_golden['text'] = df_golden['text'].astype(str)

train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])
golden_dataset = Dataset.from_pandas(df_golden[['text', 'label']])

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size (noisy): {len(test_dataset)}")
print(f"Golden Standard size: {len(golden_dataset)}")

## Metrics and Custom Trainer

In [ ]:
# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1).numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {"macro_precision": precision, "macro_recall": recall, "macro_f1": f1}

# Custom Trainer to handle class imbalance
class ImbalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = torch.tensor([4.0, 1.0]).to(model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## XLM-RoBERTa Training and Evaluation

In [ ]:
MODEL_XLM = "xlm-roberta-base"
print(f"Initializing {MODEL_XLM}...")

tokenizer_xlm = AutoTokenizer.from_pretrained(MODEL_XLM)
def tokenize_xlm(examples):
    return tokenizer_xlm(examples["text"], padding="max_length", truncation=True, max_length=128)

tok_train_xlm = train_dataset.map(tokenize_xlm, batched=True)
tok_test_xlm = test_dataset.map(tokenize_xlm, batched=True)
tok_golden_xlm = golden_dataset.map(tokenize_xlm, batched=True)

model_xlm = AutoModelForSequenceClassification.from_pretrained(MODEL_XLM, num_labels=2)

training_args_xlm = TrainingArguments(
    output_dir="./results_xlm",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer_xlm = ImbalancedTrainer(
    model=model_xlm,
    args=training_args_xlm,
    train_dataset=tok_train_xlm,
    eval_dataset=tok_test_xlm,
    compute_metrics=compute_metrics
)

print("Starting XLM-RoBERTa training...")
trainer_xlm.train()

print("\n=== XLM-RoBERTa RESULTS ON GOLDEN STANDARD ===")
preds_xlm = trainer_xlm.predict(tok_golden_xlm)
predictions_xlm = torch.argmax(torch.tensor(preds_xlm.predictions), dim=-1).numpy()
print(classification_report(preds_xlm.label_ids, predictions_xlm, target_names=['0 (Negative)', '1 (Positive)']))

## mBERT Training and Evaluation

In [ ]:
MODEL_MBERT = "bert-base-multilingual-cased"
print(f"Initializing {MODEL_MBERT}...")

tokenizer_mbert = AutoTokenizer.from_pretrained(MODEL_MBERT)
def tokenize_mbert(examples):
    return tokenizer_mbert(examples["text"], padding="max_length", truncation=True, max_length=128)

tok_train_mbert = train_dataset.map(tokenize_mbert, batched=True)
tok_test_mbert = test_dataset.map(tokenize_mbert, batched=True)
tok_golden_mbert = golden_dataset.map(tokenize_mbert, batched=True)

model_mbert = AutoModelForSequenceClassification.from_pretrained(MODEL_MBERT, num_labels=2)

training_args_mbert = TrainingArguments(
    output_dir="./results_mbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer_mbert = ImbalancedTrainer(
    model=model_mbert,
    args=training_args_mbert,
    train_dataset=tok_train_mbert,
    eval_dataset=tok_test_mbert,
    compute_metrics=compute_metrics
)

print("Starting mBERT training...")
trainer_mbert.train()

print("\n=== mBERT RESULTS ON GOLDEN STANDARD ===")
preds_mbert = trainer_mbert.predict(tok_golden_mbert)
predictions_mbert = torch.argmax(torch.tensor(preds_mbert.predictions), dim=-1).numpy()
print(classification_report(preds_mbert.label_ids, predictions_mbert, target_names=['0 (Negative)', '1 (Positive)']))